In [72]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import numpy as np

import math
import cmath
np.set_printoptions(precision = 5)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<Token var=<ContextVar name='format_options' default={'edgeitems': 3, 'threshold': 1000, 'floatmode': 'maxprec', 'precision': 8, 'suppress': False, 'linewidth': 75, 'nanstr': 'nan', 'infstr': 'inf', 'sign': '-', 'formatter': None, 'legacy': 9223372036854775807, 'override_repr': None} at 0x105e38fe0> at 0x127422f80>

In [52]:
import matplotlib
import dizx
from dizx import Edge
from dizx import clifford_simplifier as simp
Phase = dizx.CliffordPhase
d = 3
Z = dizx.VertexType.Z
X = dizx.VertexType.X

In [53]:
c = dizx.Circuit(qudit_amount=2,dim=3)
c.add_gates("H H S",0)
# c.add_gate("CZ",0,1)
# c.add_gates("S S H",1)
# c.add_gates("H S", 0)

In [54]:
print(c)

Circuit[3](2 qudits, 0 dits, [HAD(0), HAD(0), S(0)] gates)


In [55]:
# Experimenting with sympy's finite fields over polynomials
# Attempt 1 using Polynomial rings

from sympy import symbols, groebner, Matrix
from sympy.polys.domains import GF, PolynomialRing
from sympy.polys.matrices import DomainMatrix

p = 7
F = GF(p)
beta = symbols("beta")
b = symbols("b")
a = symbols("a")
ainv = symbols("a^{-1}")

# Polynomial ring GF(p)[a]
R = PolynomialRing(F, [a,ainv,b])

# testing
# R = PolynomialRing(F, [a,ainv,b,beta])

Ra = R.convert(a)
Rainv = R.convert(ainv)
# Rbeta = R.convert(beta)
Rb = R.convert(b)

G = groebner([a*ainv - 1], a, ainv, modulus=p)

H = DomainMatrix([[R.convert(0), R.convert(-1)],
                  [R.convert(1), R.convert(0)]],
                 (2, 2), R)

Hinv = DomainMatrix([[R.convert(0), R.convert(1)],
                  [R.convert(-1), R.convert(0)]],
                 (2, 2), R)


def S_repetitions(rep):
    return DomainMatrix([[R.convert(1), R.convert(0)],
                  [rep, R.convert(1)]],
                 (2, 2), R)

# Sbeta = S_repetitions(Rbeta)
Sb = S_repetitions(Rb)
# Build a matrix with entries in GF(p)[a]
A = DomainMatrix([[R.convert(1), R.convert(0)],
                  [R.convert(b), R.convert(1)]],
                 (2, 2), R)

print("Matrix over GF(p)[a]:")
print(A.to_Matrix())

aval = F(3)  # element of GF(7)

# Substitute 'a=3' in each entry
# evaluated_entries = [[entry.evaluate(0,aval) for entry in row]
#                      for row in A.to_list()]

# # Build a new matrix over GF(p)
# A_eval = DomainMatrix(evaluated_entries, A.shape, F)

# print("Matrix evaluated at a=3 (in GF(7)):")
# print(A_eval.to_Matrix())

Matrix over GF(p)[a]:
Matrix([[1, 0], [b, 1]])


In [73]:
# Attempt 2 using groebner bases to automate the reductions of the variables and say a*ainv = 1.
G = groebner([a*ainv - 1], a, ainv, b, modulus=p)
expr = a**2*ainv + 3
G.reduce(expr)

H = Matrix([[0,-1],[1,0]])
Hinv = Matrix([[0,1],[-1,0]])
H2 = Matrix([[-1,0],[0,-1]])


def Sr(rep):
    return Matrix([[1,0],[rep,1]])

M = H*Sr(-a*(b+1))*Hinv*Sr(-ainv)*Hinv*Sr(-a)*Hinv

def reduce_matrix(M):
    evaluated_entries = []
    for i in range(M.rows):
        row = M.row(i)
        newrow = []
        for entry in row:
            entry = G.reduce(entry)[1]
            newrow.append(entry)
        evaluated_entries.append(newrow)
    return Matrix(evaluated_entries)

reduce_matrix(M)

Matrix([
[     b, -a],
[a^{-1},  0]])

# The Qupit Equality Checking

In [79]:
from sympy import Matrix, symbols, expand
from sympy.polys import groebner

# symbols and field
a, ainv, b = symbols('a ainv b')
p = 101  # your modulus

# ideal: a*ainv = 1 over GF(p)
G = groebner([a*ainv - 1], a, ainv, b, order='lex', modulus=p)

def nf(expr):
    # reduced (normal-form) representative modulo <a*ainv-1> over GF(p)
    return G.reduce(expand(expr))[1]

def normalize_matrix(M):
    # apply reduction entrywise
    return M.applyfunc(nf)

def matrices_equal(A, B):
    # equal iff their difference reduces to 0
    R = normalize_matrix(A - B)
    return R == Matrix.zeros(*A.shape)

# example usage:
# M = ... (your matrix)
# N = ... (another matrix built the same way)
# are_equal = matrices_equal(M, N)
# print(are_equal)


# The Qutrit Equality Checking

In [77]:
def equalUpToScalar(U, V, tol=1e-10):
    """
    Check if two unitary matrices U and V are equal up to (-w)^t, 
    where w is the third root of unity and t is in Z6.

    Parameters:
    U : np.ndarray
        The first symplectic matrix
    V : np.ndarray
        The second symplectic matrix
    tol : float
        The numerical tolerance for equality checks (default: 1e-10).

    Returns:
    bool : True if U and V are equal up to a scalar, False otherwise.
    """
    # Compute the third root of unity w
    # w = np.exp(2j * np.pi / 3)

    # Define the all-zero matrix based on the dimension and data type of the input matrix
    # zero_matrix = np.zeros_like(U)
    zero_matrix = Matrix([[0,0],[0,0]])
    
    # When V = U
    if np.allclose(V - U, zero_matrix, rtol=0, atol=tol):
        print("Exactly equal")
        return True
    else:
        print("Not equal")
        return False
    

In [49]:
from sympy import Matrix

M = Matrix([[1,0],[a,1]])
M.inv()

Matrix([
[ 1, 0],
[-a, 1]])

In [58]:
# Test 1
H2 = H * H
print(H2)

H3 = H * H * H
print(H3)

H4 = H * H * H * H
print(H4)

Matrix([[-1, 0], [0, -1]])
Matrix([[0, 1], [-1, 0]])
Matrix([[1, 0], [0, 1]])


In [66]:
# Test 2
S = Sr(1)
print("S: ", S)

S2 = Sr(2)
reduce_S2 = reduce_matrix(S2)
print("S2: ", reduce_S2)

S3 = Sr(3)
reduce_S3 = reduce_matrix(S3)
print("S3: ", reduce_S3)

S4 = Sr(4)
reduce_S4 = reduce_matrix(S4)
print("S4: ", reduce_S4)

S5 = Sr(5)
reduce_S5 = reduce_matrix(S5)
print("S5: ", reduce_S5)

S6 = Sr(6)
reduce_S6 = reduce_matrix(S6)
print("S6: ", reduce_S6)

S7 = Sr(7)
reduce_S7 = reduce_matrix(S7)
print("S7: ", reduce_S7)

S8 = Sr(8)
reduce_S8 = reduce_matrix(S8)
print("S8: ", reduce_S8)

S:  Matrix([[1, 0], [1, 1]])
S2:  Matrix([[1, 0], [2, 1]])
S3:  Matrix([[1, 0], [3, 1]])
S4:  Matrix([[1, 0], [-3, 1]])
S5:  Matrix([[1, 0], [-2, 1]])
S6:  Matrix([[1, 0], [-1, 1]])
S7:  Matrix([[1, 0], [0, 1]])
S8:  Matrix([[1, 0], [1, 1]])


In [83]:
# Test 3: Figure 1, single qupit
LHS = H * S * H * S * H * S
RHS = Matrix([[1,0],[0,1]])
print(LHS)
print(RHS)
print(matrices_equal(LHS, RHS))


LHS = H2 * S
RHS = S * H2
print(LHS)
print(RHS)
print(matrices_equal(LHS, RHS))

Matrix([[1, 0], [0, 1]])
Matrix([[1, 0], [0, 1]])
True
Matrix([[-1, 0], [-1, -1]])
Matrix([[-1, 0], [-1, -1]])
True


In [85]:
# Test 4
input = Matrix([[a],[0]])
print(H * input)

input = Matrix([[0],[a]])
print(H * input)

input = Matrix([[0],[-a]])
print(H * input)

Matrix([[0], [a]])
Matrix([[-a], [0]])
Matrix([[a], [0]])


In [ ]:
M = H*Sb*Hinv*Sbeta*Hinv
print(M)

TypeError: cannot unpack non-iterable int object

In [35]:
M*DomainMatrix([[0],[Rb]],(2,1),R)

TypeError: cannot unpack non-iterable int object

In [37]:
Aab = H*S_repetitions(-Ra*(Rb+1))*Hinv*S_repetitions(-Rainv)*Hinv*S_repetitions(-Ra)*Hinv
Aab

TypeError: cannot unpack non-iterable int object

In [ ]:
def build_A_matrix(a,b):
    return DomainMatrix([[b, a],
                         [-a, b]],
                         (2, 2), R)

A = build_A_matrix(R.convert(1),R.convert(2))
Omega = DomainMatrix([[R.convert(0), R.convert(1)],
                  [R.convert(-1), R.convert(0)]],
                 (2, 2), R)

A.transpose()*Omega*A

In [ ]:
def eval_matrix(M, values):
    evaluated_entries = []
    for row in M.to_list():
        for entry in row:
            for key,value in values.items():
                entry = entry.evaluate(0)
    [[entry.evaluate() for entry in row]
                      for row in M.to_list()]

In [ ]:
cs = simp.CliffordSimplifier(c)

In [ ]:
cs.dag.children[0].children[0].children

In [ ]:
cs.combine_gates()

In [ ]:
cs.circuit

In [ ]:
cs.dag.children[0].children

In [ ]:
cs.remove_identity_gate()

In [ ]:
cs.push_pauli()

In [ ]:
cs.push_double_hadamard()

In [ ]:
cs.circuit

In [ ]:
cs.dag.children[0].children

In [ ]:
lhs = dizx.Circuit(qudit_amount=1, dim=3)
lhs.add_gates("S X H S H H", 0)
rhs = dizx.Circuit(qudit_amount=1, dim=3)
rhs.add_gates("H S S H", 0)


In [ ]:
c = lhs + rhs.adjoint()
c

In [ ]:
cs = simp.CliffordSimplifier(c)

In [ ]:
cs.simple_optimize()
cs.circuit

In [ ]:
for circ in cs.circuit_list:
    print(circ)

In [ ]:
cs.euler_decomp()

In [ ]:
cs.euler_decomp2()

In [ ]:
cs.dag

In [ ]:
cs.circuit

In [ ]:
cs.simple_optimize()
cs.circuit

In [ ]:
for circ in cs.circuit_list:
    print(circ)

In [ ]:
i = 0
dag = cs.dag.children[0]
while i<10:
    if len(dag.children) == 0: break
    print("Num children: ", len(dag.children))
    print("First child: ", dag.children[0].node)
    dag = dag.children[0]
    i += 1

In [ ]:
cs.push_pauli()
cs.circuit

In [ ]:
cs.combine_gates()
cs.circuit

In [ ]:
cs.remove_identity_gate()
cs.circuit

In [ ]:
cs.circuit

In [ ]:
cs.push_pauli()

In [ ]:
cs.circuit

In [ ]:
cs.push_pauli()

In [ ]:
cs.circuit

In [ ]:
print(cs.push_pauli())
cs.circuit

In [ ]:
print(cs.push_pauli())
cs.circuit

In [ ]:
print(cs.push_pauli())
cs.circuit

In [ ]:
print(cs.combine_gates())
cs.circuit

In [ ]:
print(cs.remove_identity_gate())
cs.circuit